# Lab 2: Data Validation and Baseline Model

Dataset: UNSW-NB15 (15,000 rows stratified sample)

Tasks:
1. Verify dataset integrity (SHA256)
2. Check for missing values, duplicates, and data types
3. Split into train/test with stratification
4. Train baseline model
5. Evaluate metrics

In [1]:

import hashlib
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports loaded successfully")


Imports loaded successfully


In [2]:

DATA_PATH = Path('../data/dataset.csv')

# Verify SHA256
file_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
expected_hash = '5a9ca6fa1d14d68d17f35fc4a942c68d061f90bb26bf43c1f308ff9bdc9698f6'

print(f"SHA256: {file_hash}")
print(f"Expected: {expected_hash}")
print(f"Match: {file_hash == expected_hash}")

assert file_hash == expected_hash, "Dataset integrity check failed!"
print("✓ Dataset integrity verified")


SHA256: 5a9ca6fa1d14d68d17f35fc4a942c68d061f90bb26bf43c1f308ff9bdc9698f6
Expected: 5a9ca6fa1d14d68d17f35fc4a942c68d061f90bb26bf43c1f308ff9bdc9698f6
Match: True
✓ Dataset integrity verified


In [3]:

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst 3 rows:")
display(df.head(3))
print(f"\nBasic statistics:")
display(df.describe())


Shape: (15000, 36)

Columns: ['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'attack_cat', 'label']

Data types:
dur                  float64
proto                    str
service                  str
state                    str
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dt

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000002,udp,-,INT,2,0,1064,0,500000.00000,2.128000e+09,...,0,0,1,1,0,0,0,0,Normal,0
1,0.188067,tcp,http,FIN,10,12,838,8514,111.66233,3.211621e+04,...,1,3755,1,1,0,0,1,0,Exploits,1
2,0.000000,arp,-,INT,1,0,46,0,0.00000,0.000000e+00,...,0,0,1,1,0,0,0,1,Normal,0



Basic statistics:


,dur,spkts,dpkts,sbytes,dbytes,rate,sload,dload,sloss,dloss,...,dmean,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,label
count,15000.000000,15000.000000,15000.000000,1.500000e+04,1.500000e+04,15000.000000,1.500000e+04,1.500000e+04,15000.000000,15000.000000,...,15000.000000,15000.000000,1.500000e+04,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.00000
mean,0.971213,18.577933,16.217800,8.225926e+03,1.153870e+04,81869.726917,6.331990e+07,6.233310e+05,4.817000,5.679067,...,114.179667,0.091867,1.490775e+03,4.924467,3.654400,0.007867,0.007933,0.125800,0.010800,0.55060
std,4.553601,150.804852,72.013021,1.961577e+05,8.994938e+04,145128.092872,1.743743e+08,2.378376e+06,73.610338,32.340892,...,242.457848,0.296141,3.075600e+04,8.362063,5.871507,0.088348,0.089467,0.626099,0.103364,0.49745
min,0.000000,1.000000,0.000000,2.400000e+01,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,0.000008,2.000000,0.000000,1.140000e+02,0.000000e+00,28.792308,1.122403e+04,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,0.013972,6.000000,2.000000,5.340000e+02,1.780000e+02,2626.972300,5.748031e+05,2.076388e+03,1.000000,0.000000,...,44.000000,0.000000,0.000000e+00,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.00000
75%,0.717011,12.000000,10.000000,1.244000e+03,8.480000e+02,111111.110000,6.514286e+07,1.428233e+04,3.000000,2.000000,...,84.000000,0.000000,0.000000e+00,4.000000,3.000000,0.000000,0.000000,0.000000,0.000000,1.00000
max,59.999990,10646.000000,3128.000000,1.435577e+07,4.252752e+06,1000000.000000,4.256000e+09,2.023247e+07,5319.000000,1561.000000,...,1471.000000,8.000000,2.063451e+06,59.000000,37.000000,1.000000,2.000000,16.000000,1.000000,1.00000


In [4]:

# Missing values
missing = df.isna().sum().sum()
print(f"Total missing values: {missing}")
if missing > 0:
    print("\nMissing values per column:")
    display(df.isna().sum()[df.isna().sum() > 0])

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Data types check
print(f"\nNumeric columns: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"Categorical columns: {len(df.select_dtypes(include=['object']).columns)}")

# Target distribution
print(f"\nTarget distribution (label):")
print(df['label'].value_counts())
print(f"\nTarget distribution (percentages):")
print((df['label'].value_counts(normalize=True) * 100).round(2))


Total missing values: 0



Duplicate rows: 4281

Numeric columns: 32
Categorical columns: 4

Target distribution (label):
label
1    8259
0    6741
Name: count, dtype: int64

Target distribution (percentages):
label
1    55.06
0    44.94
Name: proportion, dtype: float64


/tmp/ipykernel_7268/647390035.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(f"Categorical columns: {len(df.select_dtypes(include=['object']).columns)}")
